# Verified tool calls for an external model

OSS tutorial · synthetic data · no model call by default. The external model proposes tool arguments; the host validates and executes a fixed function. FactCrypt extends this pattern when even the values must remain private.

In [ ]:
%pip install -q "trustchain>=3.3.0,<4"

In [ ]:
import json
from trustchain.v2.signer import Signer
from trustchain.receipt import Receipt, build_receipt
signer = Signer()
pinned_key = signer.get_public_key()

In [ ]:
import ast
from decimal import Decimal, localcontext

def calculate(expression):
    if not isinstance(expression, str) or len(expression) > 256:
        raise ValueError("Expression too large")
    tree = ast.parse(expression, mode="eval")
    if sum(1 for _ in ast.walk(tree)) > 64:
        raise ValueError("Too many operations")
    def visit(node, depth=0):
        if depth > 16:
            raise ValueError("Expression too deep")
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            value = Decimal(ast.get_source_segment(expression, node))
        elif isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.UAdd, ast.USub)):
            value = visit(node.operand, depth+1)
            if isinstance(node.op, ast.USub): value = -value
        elif isinstance(node, ast.BinOp):
            a, b = visit(node.left, depth+1), visit(node.right, depth+1)
            if isinstance(node.op, ast.Add): value = a+b
            elif isinstance(node.op, ast.Sub): value = a-b
            elif isinstance(node.op, ast.Mult): value = a*b
            elif isinstance(node.op, ast.Div): value = a/b
            elif isinstance(node.op, ast.Pow) and b == int(b) and 0 <= b <= 12: value = a**int(b)
            else: raise ValueError("Operator forbidden")
        else:
            raise ValueError("Only bounded arithmetic is supported")
        if not value.is_finite() or abs(value) > Decimal("1e12"):
            raise ValueError("Result out of range")
        return value
    with localcontext() as ctx:
        ctx.prec = 30
        return str(visit(tree.body))
assert calculate("2 ** 10") == "1024"
try:
    calculate("__import__('os').getcwd()")
except ValueError:
    pass
else:
    raise AssertionError("Arbitrary code must be rejected")

## Validate the requested tool and arguments

A signed result does not make arbitrary code safe. Only the declared calculator is callable; unsupported fields and expressions are rejected.

In [ ]:
def execute_proposal(proposal):
    if type(proposal) is not dict or set(proposal) != {"tool", "arguments"}:
        raise ValueError("Unexpected proposal")
    if proposal["tool"] != "calculate" or type(proposal["arguments"]) is not dict:
        raise ValueError("Unknown tool")
    if set(proposal["arguments"]) != {"expression"}:
        raise ValueError("Unexpected arguments")
    result = calculate(proposal["arguments"]["expression"])
    signed = signer.sign("calculate", {"result": result})
    return build_receipt(signed, pinned_key, key_id=signer.get_key_id()).to_dict()

# Deterministic input fixture: this is not an LLM response.
receipt = execute_proposal({"tool":"calculate", "arguments":{"expression":"(12.5 * 8) + 3"}})
assert Receipt.load(receipt).verify(expected_public_key_b64=pinned_key).valid
assert receipt["envelope"]["data"]["result"] == "103.0"
print({"mode":"offline fixture", "verified":True, "result":receipt["envelope"]["data"]["result"]})

## Optional OpenRouter call

Off by default. Set OPENROUTER_API_KEY and OPENROUTER_MODEL outside the notebook, install the provider's compatible client, and explicitly enable the next cell. Never paste credentials into a cell or its output. The notebook sends only a synthetic arithmetic task.

In [ ]:
import os
RUN_OPENROUTER = False
if RUN_OPENROUTER:
    from openai import OpenAI
    key, model = os.environ.get("OPENROUTER_API_KEY"), os.environ.get("OPENROUTER_MODEL")
    if not key or not model:
        raise RuntimeError("Provider credentials and model must be configured outside the notebook")
    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)
    response = client.chat.completions.create(model=model, messages=[
        {"role":"system","content":'Return only JSON: {"tool":"calculate","arguments":{"expression":"..."}}. Use only bounded arithmetic.'},
        {"role":"user","content":"Calculate 12.5 times 8 plus 3."},
    ], max_tokens=200)
    proposal = json.loads(response.choices[0].message.content)
    receipt = execute_proposal(proposal)
    assert Receipt.load(receipt).verify(expected_public_key_b64=pinned_key).valid
    print({"mode":"live provider", "verified":True})
else:
    print("Live provider step not run")

## Confidential values

For confidential typed facts and application-defined calculations use [FactCrypt (Pro)](https://petro1eum.github.io/trust_chain/factcrypt-demo.html): the model proposes formulas over references and the owner computes locally. The external model receives neither plaintext facts nor decryption keys.